# MCP and LangChain tool integration (Chapter 8)

This notebook connects to the included `mcp_server.py`, converts its MCP tools with `dspy.Tool.from_mcp_tool`, and demonstrates `dspy.Tool.from_langchain`.

**Required environment variable:** `OPENAI_API_KEY`.

MCP and converted LangChain tools are asynchronous, so the notebook uses top-level `await` with `agent.acall(...)`.


In [ ]:
%pip install -r ../requirements.txt -q


In [ ]:
%pip install mcp langchain-core -q


In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))


## Connect to a local MCP server


In [ ]:
import sys
from pathlib import Path

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client


def find_server_script() -> Path:
    candidates = [Path("mcp_server.py"), Path("chapter08/mcp_server.py")]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError("Could not find chapter08/mcp_server.py")


async def build_agent():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=[str(find_server_script())],
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            mcp_tools = await session.list_tools()
            tools = [
                dspy.Tool.from_mcp_tool(session, tool)
                for tool in mcp_tools.tools
            ]

            agent = dspy.ReAct(
                "user_request -> response",
                tools=tools,
                max_iters=10,
            )

            return await agent.acall(
                user_request=(
                    "Find a flight from SFO to JFK on September 1st. "
                    "Do not book it."
                )
            )


result = await build_agent()
print(result.response)


## Convert a LangChain tool


In [ ]:
from langchain_core.tools import tool as lc_tool


@lc_tool
def web_search(query: str) -> str:
    """Search the web for information."""
    return f"[stub] web hits for: {query}"


dspy_tool = dspy.Tool.from_langchain(web_search)
langchain_agent = dspy.ReAct(
    "question -> answer",
    tools=[dspy_tool],
    max_iters=4,
)

langchain_result = await langchain_agent.acall(
    question="What does DSPy optimize?"
)
print(langchain_result.answer)


A synchronous `agent(...)` call with an asynchronous converted tool raises `ValueError` by default. Prefer `await agent.acall(...)`; enable `dspy.configure(allow_tool_async_sync_conversion=True)` only when you deliberately want DSPy to bridge the two execution modes.
